# Pain Expression Detection — MobileNetV2 Fine-Tuning
**Datasets:**
- `Dynamic_faces` — short videos; middle frame extracted per video → pain labels via emotion mapping
- `Emotional_faces` — static images per subject per emotion → pain labels via emotion mapping

**Emotion → Pain Level mapping:**
| Emotion | Pain Level |
|---|---|
| neutral, happiness | No_Pain |
| sadness | Mild |
| fear, surprise | Moderate |
| anger, disgust | Severe |

**Approach:** Transfer learning — fine-tune MobileNetV2 pre-trained on ImageNet  
**Split:** 70% train / 15% val / 15% test (explicit, stratified)  
**Training:** 2-phase — head only (epochs 1-3) → full network (epochs 4-10)  
**Imbalance:** Weighted CrossEntropyLoss (Severe class over-represented)

---
### Steps
1. Mount Google Drive → unzip dataset
2. Load + split dataset from both sources
3. Augmentation + DataLoaders
4. MobileNetV2 fine-tuning with weighted loss
5. Checkpoint saving every 2 epochs
6. Evaluate on test set → accuracy, precision, recall, F1, confusion matrix

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
import os, zipfile

# ── CHANGE THIS to your zip file path in Google Drive ────────────────────────
DRIVE_ZIP   = '/content/drive/MyDrive/pain_dataset_new.zip'
EXTRACT_DIR = '/content/pain_dataset'
# ─────────────────────────────────────────────────────────────────────────────

# Inspect zip contents before extracting
print('Zip contents (top 20 entries):')
with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
    for name in list(z.namelist())[:20]:
        print(' ', name)

# Extract
if not os.path.exists(EXTRACT_DIR):
    print('\nExtracting...')
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print('Done.')
else:
    print('\nAlready extracted.')

# Show folder tree (3 levels deep)
print('\nExtracted structure:')
for root, dirs, files in os.walk(EXTRACT_DIR):
    level = root.replace(EXTRACT_DIR, '').count(os.sep)
    if level > 2:
        continue
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level == 2:
        print(f'{indent}  ({len(files)} files)')


In [ ]:
import os, json, copy, time
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import MobileNet_V2_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

print(f'PyTorch  : {torch.__version__}')
print(f'GPU      : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name : {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────

BASE_DIR      = Path(EXTRACT_DIR)

# Path to extracted Dynamic_faces frames  (images/extracted_frames/<PainLevel>/)
FRAMES_DIR    = BASE_DIR / 'images' / 'extracted_frames'

# Path to Emotional_faces static images  (images/Emotional_faces/Emotional_faces/<subject>/<emotion>.jpg)
EMOTIONAL_DIR = BASE_DIR / 'images' / 'Emotional_faces' / 'Emotional_faces'

print(f'Frames dir    : {FRAMES_DIR}   exists={FRAMES_DIR.exists()}')
print(f'Emotional dir : {EMOTIONAL_DIR}  exists={EMOTIONAL_DIR.exists()}')

CLASSES      = ['No_Pain', 'Mild', 'Moderate', 'Severe']
NUM_CLASSES  = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

EMOTION_TO_PAIN = {
    'neutral':   'No_Pain',
    'happiness': 'No_Pain',
    'hapiness':  'No_Pain',   # dataset typo
    'sadness':   'Mild',
    'fear':      'Moderate',
    'surprise':  'Moderate',
    'suprise':   'Moderate',  # dataset typo
    'surpris':   'Moderate',  # dataset typo
    'anger':     'Severe',
    'disgust':   'Severe',
    'disgest':   'Severe',    # dataset typo
}

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

# Split
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# Training
BATCH_SIZE       = 32
NUM_EPOCHS       = 10
LR_HEAD          = 1e-3
LR_FINETUNE      = 1e-4
PHASE1_EPOCHS    = 3
CHECKPOINT_EVERY = 2
SEED             = 42

SAVE_DIR = Path('/content/drive/MyDrive/pain_model_weights')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Device     : {device}')
print(f'Classes    : {CLASSES}')
print(f'Save dir   : {SAVE_DIR}')


In [ ]:
# ── Load image paths + labels from BOTH datasets ──────────────────────────────

all_paths, all_labels = [], []

# ── Source 1: Dynamic_faces extracted frames ──────────────────────────────────
# Structure: extracted_frames/<PainLevel>/*.jpg
dyn_counts = {}
if FRAMES_DIR.exists():
    for pain_level in CLASSES:
        label_dir = FRAMES_DIR / pain_level
        if not label_dir.exists():
            continue
        imgs = sorted([p for p in label_dir.iterdir() if p.suffix.lower() in IMG_EXTS])
        all_paths.extend(imgs)
        all_labels.extend([CLASS_TO_IDX[pain_level]] * len(imgs))
        dyn_counts[pain_level] = len(imgs)
    print('Dynamic_faces (extracted frames):')
    for lbl, cnt in dyn_counts.items():
        print(f'  {lbl:<12} {cnt}')
else:
    print('WARNING: extracted_frames/ not found — run extract_frames.py first')

# ── Source 2: Emotional_faces static images ───────────────────────────────────
# Structure: Emotional_faces/Emotional_faces/<subject>/<emotion>.jpg
emo_counts = {}
if EMOTIONAL_DIR.exists():
    for subject_dir in sorted(EMOTIONAL_DIR.iterdir()):
        if not subject_dir.is_dir():
            continue
        for img_path in sorted(subject_dir.iterdir()):
            if img_path.suffix.lower() not in IMG_EXTS:
                continue
            emotion    = img_path.stem.lower()
            pain_level = EMOTION_TO_PAIN.get(emotion)
            if pain_level is None:
                continue
            all_paths.append(img_path)
            all_labels.append(CLASS_TO_IDX[pain_level])
            emo_counts[pain_level] = emo_counts.get(pain_level, 0) + 1
    print('Emotional_faces (static images):')
    for lbl in CLASSES:
        print(f'  {lbl:<12} {emo_counts.get(lbl, 0)}')
else:
    print('WARNING: Emotional_faces/ not found')

# ── Combined totals ───────────────────────────────────────────────────────────
total_counts = Counter(all_labels)
print(f'\nCombined total: {len(all_paths)} images')
for i, cls in enumerate(CLASSES):
    print(f'  {cls:<12} {total_counts[i]}')

# ── Explicit Train / Val / Test split (stratified) ────────────────────────────
X_trainval, X_test, y_trainval, y_test = train_test_split(
    all_paths, all_labels,
    test_size=TEST_RATIO,
    stratify=all_labels,
    random_state=SEED
)

val_frac = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=val_frac,
    stratify=y_trainval,
    random_state=SEED
)

print(f'\nTrain : {len(X_train)} images  ({len(X_train)/len(all_paths)*100:.1f}%)')
print(f'Val   : {len(X_val)}  images  ({len(X_val)/len(all_paths)*100:.1f}%)')
print(f'Test  : {len(X_test)}  images  ({len(X_test)/len(all_paths)*100:.1f}%)')

# Verify class balance per split
for name, lbls in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    c    = Counter(lbls)
    line = '  '.join(f'{CLASSES[i]}:{c[i]}' for i in range(NUM_CLASSES))
    print(f'{name:<6}: {line}')

# ── Compute class weights for imbalanced dataset ──────────────────────────────
train_counter = Counter(y_train)
total_train   = len(y_train)
class_weights = torch.tensor(
    [total_train / (NUM_CLASSES * train_counter[i]) for i in range(NUM_CLASSES)],
    dtype=torch.float32
)
print(f'\nClass weights (for loss): {[round(w.item(),3) for w in class_weights]}')

In [ ]:
# ── Transforms ────────────────────────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('Train augmentations:')
for t in train_transform.transforms:
    print(f'  {t.__class__.__name__}')

In [ ]:
# ── Dataset class + DataLoaders ───────────────────────────────────────────────
class PainDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths     = paths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


train_ds = PainDataset(X_train, y_train, train_transform)
val_ds   = PainDataset(X_val,   y_val,   val_transform)
test_ds  = PainDataset(X_test,  y_test,  val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Test  batches : {len(test_loader)}')

In [ ]:
# ── Visualise sample training images ──────────────────────────────────────────
COLORS = {'No_Pain': '#2ecc71', 'Mild': '#f1c40f', 'Moderate': '#e67e22', 'Severe': '#e74c3c'}

def denorm(tensor):
    mean = np.array(IMAGENET_MEAN)
    std  = np.array(IMAGENET_STD)
    img  = tensor.permute(1, 2, 0).numpy()
    return (img * std + mean).clip(0, 1)

imgs, lbls = next(iter(train_loader))
fig, axes  = plt.subplots(2, 8, figsize=(18, 5))
for i, ax in enumerate(axes.flatten()):
    img  = denorm(imgs[i])
    cls  = CLASSES[lbls[i].item()]
    ax.imshow(img)
    ax.set_title(cls, fontsize=8, color=COLORS[cls], weight='bold')
    ax.axis('off')
plt.suptitle('Sample Training Images (augmented)', fontsize=13, weight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── MobileNetV2 model ─────────────────────────────────────────────────────────
def build_mobilenet(num_classes):
    model = models.mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )

    # Phase 1: freeze all feature layers except last 3 blocks
    for param in model.features.parameters():
        param.requires_grad = False
    for param in model.features[-3:].parameters():
        param.requires_grad = True

    return model


model = build_mobilenet(NUM_CLASSES).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f'Model            : MobileNetV2')
print(f'Total params     : {total_p:,}')
print(f'Trainable (ph.1) : {trainable:,}  ({100*trainable/total_p:.1f}%)')

In [ ]:
# ── Training helper functions ─────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct    += out.argmax(1).eq(labels).sum().item()
        total      += images.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        out  = model(images)
        loss = criterion(out, labels)
        total_loss += loss.item() * images.size(0)
        correct    += out.argmax(1).eq(labels).sum().item()
        total      += images.size(0)
    return total_loss / total, correct / total


def save_checkpoint(model, optimizer, epoch, val_acc, path):
    torch.save({
        'epoch':                epoch,
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_acc':              val_acc,
    }, path)
    print(f'  [Checkpoint] saved → {path.name}')

print('Helper functions defined.')

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
#
# WeightedCrossEntropyLoss compensates for Severe class being over-represented
# (anger + disgust mapped to Severe gives ~2× more Severe samples than others)

criterion      = nn.CrossEntropyLoss(weight=class_weights.to(device))
best_val_acc   = 0.0
best_wts       = copy.deepcopy(model.state_dict())
history        = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD
)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

print('=' * 65)
print('  MobileNetV2 Fine-Tuning — Pain Expression Classification')
print('=' * 65)
print(f'  Phase 1 (ep 1-{PHASE1_EPOCHS}): head + last 3 blocks,  lr={LR_HEAD}')
print(f'  Phase 2 (ep {PHASE1_EPOCHS+1}-{NUM_EPOCHS}): full network,  lr={LR_FINETUNE}')
print()

for epoch in range(1, NUM_EPOCHS + 1):

    # ── Switch to Phase 2 ────────────────────────────────────────────────────
    if epoch == PHASE1_EPOCHS + 1:
        print(f'--- Phase 2 start: unfreezing full network (epoch {epoch}) ---')
        for param in model.parameters():
            param.requires_grad = True
        print(f'    Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
        optimizer = optim.Adam(model.parameters(), lr=LR_FINETUNE)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=NUM_EPOCHS - PHASE1_EPOCHS
        )

    t0 = time.time()
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    vl_loss, vl_acc = eval_epoch(model,  val_loader,   criterion)
    scheduler.step()
    elapsed = time.time() - t0

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    print(f'Ep {epoch:>2}/{NUM_EPOCHS}  '
          f'tr_loss={tr_loss:.4f} tr_acc={tr_acc:.4f}  '
          f'vl_loss={vl_loss:.4f} vl_acc={vl_acc:.4f}  '
          f'({elapsed:.0f}s)')

    if epoch % CHECKPOINT_EVERY == 0:
        ckpt = SAVE_DIR / f'checkpoint_epoch{epoch:02d}_acc{vl_acc:.3f}.pth'
        save_checkpoint(model, optimizer, epoch, vl_acc, ckpt)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        best_wts     = copy.deepcopy(model.state_dict())
        torch.save(best_wts, SAVE_DIR / 'best_model.pth')
        print(f'  ★ New best model  (val_acc={best_val_acc:.4f})')

torch.save(model.state_dict(), SAVE_DIR / 'final_model.pth')
print(f'\nFinal model  → {SAVE_DIR}/final_model.pth')
print(f'Best model   → {SAVE_DIR}/best_model.pth  (val_acc={best_val_acc:.4f})')

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
epochs = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train', linewidth=2, markersize=5)
axes[0].plot(epochs, history['val_loss'],   'r-o', label='Val',   linewidth=2, markersize=5)
axes[0].axvline(x=PHASE1_EPOCHS + 0.5, color='gray', linestyle='--', alpha=0.7, label='Phase 2 start')
axes[0].set_title('Loss per Epoch', fontsize=13, weight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, [a*100 for a in history['train_acc']], 'b-o', label='Train', linewidth=2, markersize=5)
axes[1].plot(epochs, [a*100 for a in history['val_acc']],   'r-o', label='Val',   linewidth=2, markersize=5)
axes[1].axvline(x=PHASE1_EPOCHS + 0.5, color='gray', linestyle='--', alpha=0.7, label='Phase 2 start')
axes[1].set_title('Accuracy per Epoch', fontsize=13, weight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('MobileNetV2 Training History', fontsize=14, weight='bold')
plt.tight_layout()
plt.savefig(SAVE_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {SAVE_DIR}/training_curves.png')

In [ ]:
# ── Load BEST model and evaluate on TEST set ──────────────────────────────────
model.load_state_dict(torch.load(SAVE_DIR / 'best_model.pth', map_location=device))
model.eval()
print(f'Loaded best_model.pth  (val_acc={best_val_acc:.4f})')
print(f'Evaluating on {len(test_ds)} test images...\n')

all_preds, all_true = [], []
with torch.no_grad():
    for images, labels in test_loader:
        preds = model(images.to(device)).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_true.extend(labels.numpy())

all_preds = np.array(all_preds)
all_true  = np.array(all_true)

acc  = accuracy_score(all_true, all_preds)
prec = precision_score(all_true, all_preds, average='weighted', zero_division=0)
rec  = recall_score(all_true, all_preds,    average='weighted', zero_division=0)
f1   = f1_score(all_true, all_preds,        average='weighted', zero_division=0)

print('=' * 48)
print('  TEST SET RESULTS')
print('=' * 48)
print(f'  Accuracy          : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision (wtd)   : {prec:.4f}')
print(f'  Recall    (wtd)   : {rec:.4f}')
print(f'  F1 Score  (wtd)   : {f1:.4f}')
print('=' * 48)

metrics_out = {
    'accuracy':    round(float(acc), 4),
    'precision':   round(float(prec), 4),
    'recall':      round(float(rec), 4),
    'f1_weighted': round(float(f1), 4),
    'best_val_acc': round(float(best_val_acc), 4),
}
with open(SAVE_DIR / 'test_metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print(f'\nMetrics saved → {SAVE_DIR}/test_metrics.json')

In [ ]:
# ── Per-class classification report ───────────────────────────────────────────
report = classification_report(all_true, all_preds, target_names=CLASSES, digits=4)
print('Per-class Classification Report:')
print(report)

with open(SAVE_DIR / 'classification_report.txt', 'w') as f:
    f.write(report)
print(f'Saved → {SAVE_DIR}/classification_report.txt')

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
cm   = confusion_matrix(all_true, all_preds)
norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, title, fmt in zip(
    axes,
    [cm,  norm],
    ['Confusion Matrix (counts)', 'Confusion Matrix (normalised)'],
    ['d', '.2f'],
):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Purples',
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax,
                linewidths=0.4, linecolor='#333')
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('True',      fontsize=10)
    ax.set_title(title,        fontsize=11, weight='bold')

plt.suptitle(f'MobileNetV2 — Test Accuracy: {acc*100:.2f}%', fontsize=13, weight='bold')
plt.tight_layout()
plt.savefig(SAVE_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {SAVE_DIR}/confusion_matrix.png')

## Results Summary

### Saved files in Google Drive (`pain_model_weights/`)

| File | Description |
|---|---|
| `best_model.pth` | Weights at best validation accuracy |
| `final_model.pth` | Weights after last epoch |
| `checkpoint_epoch02_acc*.pth` | Checkpoint every 2 epochs |
| `training_curves.png` | Loss + accuracy plots |
| `confusion_matrix.png` | Test set confusion matrix |
| `test_metrics.json` | Accuracy, precision, recall, F1 |
| `classification_report.txt` | Per-class metrics |

### Dataset
- **Dynamic_faces**: middle frame extracted from each video → ~1,448 frames
- **Emotional_faces**: 840 static images (120 subjects × 7 emotions)
- **Combined**: ~2,288 images across 4 pain classes

### Emotion → Pain mapping
| Emotion | Pain Level | Reasoning |
|---|---|---|
| neutral, happiness | No_Pain | Relaxed/positive face |
| sadness | Mild | Slight distress |
| fear, surprise | Moderate | Heightened arousal, brow tension |
| anger, disgust | Severe | Intense negative expression |

### Training strategy
- **Phase 1** (epochs 1-3): Only classifier head + last 3 feature blocks, `lr=1e-3`
- **Phase 2** (epochs 4-10): Full network fine-tuned, `lr=1e-4` with cosine annealing
- **Weighted loss**: `CrossEntropyLoss(weight=...)` compensates for Severe class over-representation
- **Augmentation**: horizontal flip, rotation ±15°, color jitter, slight translation
- **Split**: 70% train / 15% val / 15% test — stratified, no data leakage